In [13]:
import torch
import numpy as np

from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from dinosaw.wrappers import get_models, ModelTypes, MODEL_NAMES
from dinosaw.utils import add_custom_font, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [3]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s', )
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")
# n_dims = 768 if '_b' in selected_model else 384
n_dims = 384

2026-07-24 11:54:02 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 11:54:02 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-07-24 11:54:03 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 11:54:03 | I | factory.py                 : 152 | Building wrapper 'dinov3_s+' on device cuda:0
2026-07-24 11:54:03 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 11:54:03 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth
2026-07-24 11:54:03 | I | wrapper.py                 :  48 | Initialized PretrainedViTWra

In [4]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

In [6]:
ramps: tuple[RampTypes, ...] = ('random',)
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {model_key: {r: [] for r in ramps} for model_key in enabled_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for model_key in enabled_models:
    features = []
    for img_file in image_files:
        img_path = f'{ds_folder}/{img_file}'
        img = Image.open(img_path).convert('RGB')
        feats = get_features(models[model_key], img, channel_last=True)
        features.append(feats)
    for i in range(n_imgs):
        feats = features[i]
        result = do_linear_probe(feats, "random", probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        ramps_to_results[model_key]["random"].append(result)

2026-07-24 11:54:10 | I | wrapper.py                 :  92 | Processing image, size: [699, 578]
2026-07-24 11:54:10 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,574,686] -> f: [1,384,41,49]
2026-07-24 11:54:10 | I | wrapper.py                 :  92 | Processing image, size: [597, 596]
2026-07-24 11:54:10 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,588,588] -> f: [1,384,42,42]
2026-07-24 11:54:10 | I | wrapper.py                 :  92 | Processing image, size: [800, 528]
2026-07-24 11:54:10 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,798] -> f: [1,384,37,57]
2026-07-24 11:54:10 | I | wrapper.py                 :  92 | Processing image, size: [631, 512]
2026-07-24 11:54:10 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,504,630] -> f: [1,384,36,45]
2026-07-24 11:54:10 | I | wrapper.py                 :  92 | Processing image, size: [700, 700]
2026-07-24 11:54:11 | I | wrapper.py                 : 1

In [7]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [9]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [21]:
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')

n_rows, n_cols = len(enabled_models), 4

W, H = 7.5, 1.5 * 2.3

w_spacing = [2, 2, 0, 1.5]
SPACE_ROW_IDXS = (4,)

FIG_B_COL_OFFSET = 1
FIG_B_W_COLS = 2
FIG_C_COL_OFFSET = FIG_B_COL_OFFSET + FIG_B_W_COLS + 2


w_spacing[FIG_C_COL_OFFSET:] = [0.7] * len(w_spacing[FIG_C_COL_OFFSET:])

fig = plt.figure(figsize=(W , H ))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.12)
colors: dict[ModelTypes, str] = {
    'dinov2_s': '#5762D5',
    'dinov3_s+': "#9399DF",
    'alibi_coco_dinov2_s': '#16ce37',
}
ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
    'random': 'Random',
}
# spacer_row = fig.add_subplot(gs[:, 5])

TITLE_PAD = 0

top_left_ramp_ax = None
for row, ramp in enumerate(["random"]*len(enabled_models)):
    h, w = 34, 34
    ramp_arr = get_ramp(ramp, h, w)
    ramp_ax = fig.add_subplot(gs[row, 0])
    ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

    mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
    add_red_square_overlay(ramp_ax, mask, 1, 1)

    ramp_ax.set_xticks([])
    ramp_ax.set_yticks([])

    ramp_ax.set_ylabel(ramp_to_title[ramp], )

    if row == 0:
        ramp_ax.set_title('Target ramp', pad=TITLE_PAD)
        top_left_ramp_ax = ramp_ax


worst_channels = []
for row, model_key in enumerate(enabled_models):
    ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET:FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[model_key]["random"])

    print(f"{ramp}: {np.argsort(-mean_channel_scores)[:3]}")
    worst_channels.extend(list(np.argsort(-mean_channel_scores)[:2]))

    ax.hlines(0, 0, n_dims, 'red', '--')

    ax.plot(mean_channel_scores, color=colors[model_key])
    ax.fill_between(
        np.arange(n_dims),
        mean_channel_scores - std_channel_scores,
        mean_channel_scores + std_channel_scores,
        color=colors[model_key],
        alpha=0.3,
    )
    ax.set_ylim(-0.1, 1)
    ax.set_xlim(0 -10, n_dims + 10)
    ax.tick_params(axis='both', labelsize=6)


    mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)


    model_name = MODEL_NAMES[model_key].replace('(COCO)', '')
    ax.set_ylabel(f"{model_name}", fontweight = "bold" if "ALiBi" in model_name else None)
    if row == 0:
        ax.set_title('Per-channel ' + r'$R^2$' +  'scores', pad=TITLE_PAD)
        mean_pred_ax.set_title('Mean prediction \n(all channels)', )
    elif row == len(ramps) - 1:
        ax.set_xlabel('Channel', )
    

    mean_pred_ax.set_ylabel(f'$R^{2}:${mean_score:.2f}', )
    mean_pred_ax.set_xticks([])
    mean_pred_ax.set_yticks([])

SAVE = True
if SAVE:
    plt.savefig('saved/S3.pdf', bbox_inches='tight', dpi=300)
    plt.close()


findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.


random: [377 201 104]
random: [261 128  94]
random: [346 149 185]
